# 01 — AFCP-EM Expert Matching Baseline (floor baseline)

> **✅ Floor baseline — Use Case 2.** Runs end-to-end on `make expdataset` outputs.
> **No LLM, no embedding model.** Compliance gate is pure-Python rule matching;
> scoring is sklearn TF-IDF + cosine similarity. The point is a reproducible
> floor, not a state-of-the-art system — see `## Notes` for what is intentionally
> out of scope.

**Inputs (produced by `make expdataset`):**
- `data/experts/curated_profiles.parquet` — 110 curated experts (KR + EN columns)
- `data/problems_external/sme_problems_v1.json` — 226 SME problems with `client_country`, `compliance_sensitivity`, `industry_tags`
- `data/experts/curated_ratings.parquet` — 7,800 3-rater ratings (0–3 Likert); a `(problem, expert)` pair is **relevant** when ≥ 2 of 3 raters gave ≥ 2 (the 1–5 wording in the original notebook spec was wrong for this dataset — see `## Notes`)
- `data/compliance/{kr,us}_standards_v1.json` — KR ITPA + US EAR/CCL controls + per-country authorization matrices
- `data/compliance/leakage_incidents_v1.json` — L1–L4 adversarial cases the gate must catch

**Note on problem-id namespace.** `data/problems.parquet` (SIRP, IDs `prob:000…`)
and the curated ratings (IDs `PROB_001…`) are **separate problem sets** with no
overlap. The ground truth lives over the SME problems (`sme_problems_v1.json`),
so that file — not `problems.parquet` — is the problem source for this notebook.

**Run order:**

```bash
make curated-experts  # 110 curated profiles
make curated-ratings  # 7,800 3-rater ratings + κ/ICC
make compliance       # KR + US governance instances (205 triples)
jupyter nbconvert --to notebook --execute notebooks/01_matching_baseline_afcp.ipynb
```


In [1]:
from __future__ import annotations

import json
import math
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

ROOT = Path.cwd().resolve()
if (ROOT / 'data' / 'experts' / 'curated_profiles.parquet').exists():
    pass
elif (ROOT.parent / 'data' / 'experts' / 'curated_profiles.parquet').exists():
    ROOT = ROOT.parent
else:
    raise SystemExit('Run `make expdataset` first to populate data/experts/ and data/compliance/.')

experts = pd.read_parquet(ROOT / 'data' / 'experts' / 'curated_profiles.parquet')
# NaN-safe text columns: '' is falsy so `a or b or ''` chains correctly (NaN is truthy).
_obj_cols = experts.select_dtypes(include=['object']).columns
experts[_obj_cols] = experts[_obj_cols].fillna('')

ratings = pd.read_parquet(ROOT / 'data' / 'experts' / 'curated_ratings.parquet')
kr_std = json.loads((ROOT / 'data' / 'compliance' / 'kr_standards_v1.json').read_text())
us_std = json.loads((ROOT / 'data' / 'compliance' / 'us_standards_v1.json').read_text())
leakage = json.loads((ROOT / 'data' / 'compliance' / 'leakage_incidents_v1.json').read_text())

sme_path = ROOT / 'data' / 'problems_external' / 'sme_problems_v1.json'
sme = json.loads(sme_path.read_text())
problems_all = sme if isinstance(sme, list) else sme.get('problems', sme.get('items', sme))

# Keep only problems that have at least one rating (so we can evaluate them).
rated_ids = set(ratings['problem_id'].unique())
problems = [p for p in problems_all if p.get('problem_id') in rated_ids]

print(f'experts:           {experts.shape}')
print(f'problems (rated):  {len(problems)} of {len(problems_all)} in sme_problems_v1.json')
print(f'ratings:           {ratings.shape}  (raters per pair: '
      f'{ratings.groupby(["problem_id","expert_id"]).size().median():.0f} median)')
print(f'KR controls:       {len(kr_std.get("technology_controls", []))}')
print(f'US controls:       {len(us_std.get("technology_controls", []))}')
print(f'leakage incidents: {len(leakage.get("incidents", []))}')


experts:           (110, 103)
problems (rated):  75 of 226 in sme_problems_v1.json
ratings:           (7800, 12)  (raters per pair: 3 median)
KR controls:       12
US controls:       8
leakage incidents: 4


/tmp/ipykernel_26297/2866029416.py:23: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  _obj_cols = experts.select_dtypes(include=['object']).columns


## 1. Compliance gate — pure-Python rule matcher

Pure-Python rule matcher (no async, no ORM, no LLM). The gate runs *before*
scoring (architectural, not post-hoc), so a BLOCK / REVIEW verdict masks the candidate out of the pool
before TF-IDF similarities are computed.

Two rule families, both directly from the standards JSON:

1. **R001 — NCT entity list (KR ITPA §33/§34).** If `expert.kr_compliance_nct_flag == True`, hard BLOCK.
2. **R002 — tech export control.** Detect controlled tech keywords in the
   problem text; look up the destination country in `country_authorization_by_jurisdiction.matrix` (KR side) and `country_group_map` (US side):
   - `BLOCK` / E1 / E2 → BLOCK
   - `RESTRICT` / D1 → REVIEW
   - `ALLOW` / A → PERMIT


In [2]:
# --- Tech-keyword lookup tables (built once from the standards JSON) -------

def _normalize(s: str) -> str:
    return str(s or '').lower().replace('_', ' ').replace('-', ' ')

KR_CONTROLS: list[dict[str, Any]] = kr_std.get('technology_controls', [])
US_CONTROLS: list[dict[str, Any]] = us_std.get('technology_controls', [])

KR_MATRIX = {
    row['destination']: row['action']
    for row in kr_std.get('country_authorization_by_jurisdiction', {}).get('matrix', [])
}
US_GROUP_MAP = us_std.get('jurisdiction_trigger_rules', {}).get('country_group_map', {})
US_BLOCK = set(US_GROUP_MAP.get('E1_EMBARGO', [])) | set(US_GROUP_MAP.get('E2_SANCTION', []))
US_REVIEW = set(US_GROUP_MAP.get('D1_CONCERN', []))


def _problem_text_for_gate(p: dict) -> str:
    parts = [
        p.get('problem_title') or '',
        p.get('problem_description') or '',
        ' '.join(p.get('industry_tags') or []),
        ' '.join(p.get('process_area') or []),
        p.get('technology_classification') or '',
    ]
    return _normalize(' '.join(str(x) for x in parts))


def _expert_touches_us(expert: dict) -> bool:
    nat = str(expert.get('kr_nationality') or expert.get('en_nationality') or '').upper()
    hist = str(expert.get('kr_work_history_countries') or '').upper()
    return nat == 'US' or 'US' in hist


def compliance_gate(
    expert: dict,
    problem: dict,
    kr_controls: list[dict] = KR_CONTROLS,
    us_controls: list[dict] = US_CONTROLS,
) -> dict:
    """Return `{decision: PERMIT|REVIEW|BLOCK, rules: [...], reason: str}`.

    Architectural — runs before scoring; BLOCK / REVIEW remove the candidate
    from the ranked pool.
    """
    # R001 — NCT hard block (S2 in the paper's scenario table)
    if bool(expert.get('kr_compliance_nct_flag') or expert.get('kr_has_nct')):
        return {
            'decision': 'BLOCK',
            'rules': ['R001_NCT'],
            'reason': 'NCT entity list — KR ITPA §33',
        }

    text = _problem_text_for_gate(problem)
    target = (problem.get('client_country') or 'KR').upper()
    matched_rules: list[str] = []
    worst = 'PERMIT'  # PERMIT < REVIEW < BLOCK

    def _escalate(level: str) -> None:
        nonlocal worst
        order = {'PERMIT': 0, 'REVIEW': 1, 'BLOCK': 2}
        if order[level] > order[worst]:
            worst = level

    # R002 — KR ITPA controls (applies when destination ≠ KR)
    for ctrl in kr_controls:
        kw = _normalize(ctrl.get('tech_keyword', ''))
        if kw and kw in text and target != 'KR':
            action = KR_MATRIX.get(target, 'ALLOW')
            if action == 'BLOCK':
                matched_rules.append(f'R002_KR_{ctrl.get("nct_code", kw)}')
                _escalate('BLOCK')
            elif action == 'RESTRICT':
                matched_rules.append(f'R002_KR_{ctrl.get("nct_code", kw)}')
                _escalate('REVIEW')

    # R002 — US EAR/CCL controls (apply when the expert is US-tied)
    if _expert_touches_us(expert):
        for ctrl in us_controls:
            kw = _normalize(ctrl.get('tech_keyword', ''))
            if kw and kw in text:
                if target in US_BLOCK:
                    matched_rules.append(f'R002_US_{ctrl.get("eccn", kw)}')
                    _escalate('BLOCK')
                elif target in US_REVIEW:
                    matched_rules.append(f'R002_US_{ctrl.get("eccn", kw)}')
                    _escalate('REVIEW')

    reason = (
        ', '.join(matched_rules)
        if matched_rules
        else 'no applicable controls'
    )
    return {'decision': worst, 'rules': matched_rules, 'reason': reason}


# Quick sanity check on the four leakage incidents
for inc in leakage.get('incidents', [])[:4]:
    pid, eid = inc['problem_id'], inc['expert_id']
    p_match = next((p for p in problems_all if p.get('problem_id') == pid), None)
    e_match = experts[experts['expert_id'] == eid]
    if p_match is None or e_match.empty:
        print(f"{inc['incident_id']}: problem={pid} expert={eid} -> (problem/expert not in current pool)")
        continue
    v = compliance_gate(e_match.iloc[0].to_dict(), p_match)
    expected = inc['expected_verdict']
    ok = '✓' if (v['decision'] == expected or (expected == 'ESCALATE' and v['decision'] in ('REVIEW', 'BLOCK'))) else '✗'
    print(f"{inc['incident_id']} ({inc['scenario_class']}): expected={expected:8s} got={v['decision']:7s} {ok}  rules={v['rules']}")


L1 (S2): expected=BLOCK    got=PERMIT  ✗  rules=[]
L2 (S2): expected=BLOCK    got=PERMIT  ✗  rules=[]
L3: problem=ADV_PROB_004 expert=EXP_039 -> (problem/expert not in current pool)
L4: problem=ADV_PROB_024 expert=EXP_049 -> (problem/expert not in current pool)


## 2. TF-IDF scoring over the gated pool

Floor baseline: build a single TF-IDF vocabulary over expert profiles, then
for each problem compute cosine similarity to every expert. Pairs that fail
`compliance_gate` are masked to `-inf` *before* sorting, so the gate's
rejections never reach the top-K.


In [3]:
def expert_text(row: pd.Series) -> str:
    parts = [
        row.get('en_profile_summary') or row.get('kr_profile_summary') or '',
        (row.get('en_tech_tags') or row.get('kr_tech_tags') or '').replace('|', ' '),
        (row.get('en_expertise_areas') or row.get('kr_expertise_areas') or '').replace('|', ' '),
        (row.get('en_process_specialization') or row.get('kr_process_specialization') or '').replace('|', ' '),
        (row.get('en_specialization') or row.get('kr_specialization') or ''),
        (row.get('en_equipment_experience') or row.get('kr_equipment_experience') or '').replace('|', ' '),
    ]
    return ' '.join(str(p) for p in parts)


def problem_text_for_match(p: dict) -> str:
    parts = [
        p.get('problem_title') or '',
        p.get('problem_description') or '',
        ' '.join(p.get('industry_tags') or []),
        ' '.join(p.get('process_area') or []),
        ' '.join(p.get('required_expertise') or []),
        ' '.join(p.get('equipment_involved') or []),
        ' '.join(p.get('symptoms') or []),
    ]
    return ' '.join(str(p) for p in parts)


expert_docs = experts.apply(expert_text, axis=1).tolist()
problem_docs = [problem_text_for_match(p) for p in problems]

vec = TfidfVectorizer(max_features=8000, ngram_range=(1, 2), lowercase=True)
X_all = vec.fit_transform(expert_docs + problem_docs)
X_exp = X_all[: len(expert_docs)]
X_prob = X_all[len(expert_docs):]

sims = cosine_similarity(X_prob, X_exp)  # [n_problems, n_experts]
print(f'similarity matrix: {sims.shape}  (mean={sims.mean():.3f}, max={sims.max():.3f})')


similarity matrix: (75, 110)  (mean=0.012, max=0.311)


## 3. Metric functions

Pure functions for ranking and compliance metrics. They operate on lists,
dicts, and sets — no DB or async dependencies.


In [4]:
def calculate_mrr_with_ground_truth(
    ranked_results: list[list],
    ground_truth: dict[str, list[str]],
) -> float:
    """MRR with explicit ground truth.

    Args:
      ranked_results: rows of `[problem_id, expert_id_1, expert_id_2, ...]`.
      ground_truth:   `problem_id -> [relevant expert_ids]`.
    """
    rr: list[float] = []
    for row in ranked_results:
        if not row:
            continue
        pid = row[0]
        ranked = row[1:]
        if pid not in ground_truth:
            continue
        relevant = set(ground_truth[pid])
        for pos, eid in enumerate(ranked, start=1):
            if eid in relevant:
                rr.append(1.0 / pos)
                break
        else:
            rr.append(0.0)
    return sum(rr) / len(rr) if rr else 0.0


def calculate_precision_at_k(ranked: list[str], relevant: set[str], k: int = 5) -> float:
    if not ranked or k <= 0:
        return 0.0
    top = ranked[:k]
    return sum(1 for e in top if e in relevant) / min(k, len(top))


def calculate_recall_at_k(ranked: list[str], relevant: set[str], k: int) -> float:
    if not relevant or not ranked or k <= 0:
        return 0.0
    top = set(ranked[:k])
    return len(top & relevant) / len(relevant)


def calculate_ndcg_at_k(ranked: list[str], rel_scores: dict[str, float], k: int = 5) -> float:
    if not ranked or k <= 0:
        return 0.0
    dcg = 0.0
    for i, eid in enumerate(ranked[:k], start=1):
        rel = rel_scores.get(eid, 0.0)
        dcg += (2 ** rel - 1) / math.log2(i + 1)
    ideal = sorted(rel_scores.values(), reverse=True)
    idcg = 0.0
    for i, rel in enumerate(ideal[:k], start=1):
        idcg += (2 ** rel - 1) / math.log2(i + 1)
    return dcg / idcg if idcg > 0 else 0.0


def calculate_leakage_rate(total_violations: int, detected: int) -> float:
    if total_violations == 0:
        return 0.0
    return max(0, total_violations - detected) / total_violations


## 4. Ground truth + orchestration loop

- **Relevance GT.** A `(problem, expert)` pair is relevant when **≥ 2 of 3 raters scored ≥ 2** on the **0–3 Likert** scale (`curated_ratings.parquet`). NDCG uses graded relevance = mean rating / 3.
- **Leakage GT.** The four incidents L1–L4 (`leakage_incidents_v1.json`) — every one whose `expected_verdict ∈ {BLOCK, ESCALATE}` is a pair the gate must catch.
- **Orchestration.** For each problem, apply the gate to every expert, mask blocked experts out of the similarity row, sort the remaining experts by descending similarity, and record the ranked list + the per-pair verdicts for compliance metrics.


In [5]:
# --- Relevance ground truth (consensus = ≥2 of 3 raters at score ≥2, on 0–3 Likert) ---
_consensus = (
    ratings.assign(_pos=(ratings['relevance_score'] >= 2).astype(int))
    .groupby(['problem_id', 'expert_id'])
    .agg(_n=('relevance_score', 'count'), _pos=('_pos', 'sum'), _mean=('relevance_score', 'mean'))
    .reset_index()
)
_consensus['is_relevant'] = _consensus['_pos'] >= 2
ground_truth_rankings: dict[str, list[str]] = {
    pid: g.loc[g['is_relevant'], 'expert_id'].tolist()
    for pid, g in _consensus.groupby('problem_id')
}
# Graded relevance for NDCG: mean rating / 3 (so max 1.0, min 0.0).
rel_scores_by_problem: dict[str, dict[str, float]] = {
    pid: dict(zip(g['expert_id'], g['_mean'] / 3.0))
    for pid, g in _consensus.groupby('problem_id')
}

# --- Leakage ground truth ---------------------------------------------------
ground_truth_violations: set[tuple[str, str]] = {
    (inc['problem_id'], inc['expert_id'])
    for inc in leakage.get('incidents', [])
    if inc.get('expected_verdict') in ('BLOCK', 'ESCALATE')
}

print(f'GT problems with ≥1 relevant expert: {sum(1 for v in ground_truth_rankings.values() if v)}')
print(f'GT leakage violations: {len(ground_truth_violations)}')

# --- Orchestration loop -----------------------------------------------------
expert_ids = experts['expert_id'].tolist()
ranked_rows: list[list[str]] = []
verdicts: list[dict[str, Any]] = []

for j, problem in enumerate(problems):
    pid = problem['problem_id']
    sim_row = sims[j].copy()

    for i, expert in experts.iterrows():
        v = compliance_gate(expert.to_dict(), problem)
        verdicts.append({
            'problem_id': pid,
            'expert_id': expert['expert_id'],
            'decision': v['decision'],
            'rules': v['rules'],
        })
        if v['decision'] == 'BLOCK':
            sim_row[i] = -np.inf  # masked out of the ranked pool

    order = np.argsort(-sim_row)
    ranked = [expert_ids[i] for i in order if np.isfinite(sim_row[i])]
    ranked_rows.append([pid] + ranked)

print(f'evaluated {len(ranked_rows)} problems, total verdicts: {len(verdicts)}')


GT problems with ≥1 relevant expert: 55
GT leakage violations: 4


evaluated 75 problems, total verdicts: 8250


## 5. Aggregate metrics

Report MRR, NDCG@5, Precision@5, Recall@10, and leakage_rate@5. The first
four cover ranking quality; leakage_rate@5 checks how many of the L1–L4
adversarial cases survive in the top-5 (target: 0.0).


In [6]:
# --- Ranking metrics --------------------------------------------------------
mrr = calculate_mrr_with_ground_truth(ranked_rows, ground_truth_rankings)

ndcg5_scores, p5_scores, r10_scores = [], [], []
for row in ranked_rows:
    pid, ranked = row[0], row[1:]
    if pid not in ground_truth_rankings:
        continue
    rel = set(ground_truth_rankings[pid])
    if not rel:
        continue
    rel_score_map = rel_scores_by_problem.get(pid, {eid: 1.0 for eid in rel})
    ndcg5_scores.append(calculate_ndcg_at_k(ranked, rel_score_map, k=5))
    p5_scores.append(calculate_precision_at_k(ranked, rel, k=5))
    r10_scores.append(calculate_recall_at_k(ranked, rel, k=10))

# --- Leakage@5 (a GT violation appearing in top-5 = gate failed + score ranked it high) ---
top5_pairs: set[tuple[str, str]] = set()
for row in ranked_rows:
    pid, ranked = row[0], row[1:]
    for eid in ranked[:5]:
        top5_pairs.add((pid, eid))

evaluated_problem_ids = {row[0] for row in ranked_rows}
testable_violations = {
    (pid, eid) for (pid, eid) in ground_truth_violations
    if pid in evaluated_problem_ids
}
leaked_in_top5 = ground_truth_violations & top5_pairs
leakage_at_5 = len(leaked_in_top5) / max(1, len(testable_violations))
# leakage_rate@5 = fraction of TESTABLE violations that the gate failed to mask AND
# the scorer ranked into top-5. L3/L4 use ADV_PROB_* problems not in this 75-problem
# pool, so they're not testable here.

# --- Compliance summary -----------------------------------------------------
verdict_counts = pd.Series([v['decision'] for v in verdicts]).value_counts()
block_rate = verdict_counts.get('BLOCK', 0) / len(verdicts)
review_rate = verdict_counts.get('REVIEW', 0) / len(verdicts)

print(f'\n=== Ranking ===\n'
      f'  MRR             = {mrr:.4f}\n'
      f'  NDCG@5  (mean)  = {np.mean(ndcg5_scores):.4f}  (n={len(ndcg5_scores)})\n'
      f'  P@5     (mean)  = {np.mean(p5_scores):.4f}\n'
      f'  R@10    (mean)  = {np.mean(r10_scores):.4f}\n'
      f'\n=== Compliance ===\n'
      f'  BLOCK rate      = {block_rate:.4f}  ({verdict_counts.get("BLOCK", 0)} / {len(verdicts)})\n'
      f'  REVIEW rate     = {review_rate:.4f}\n'
      f'  leakage GT (total / testable) = {len(ground_truth_violations)} / {len(testable_violations)}\n'
      f'  violations leaked into top-5  = {len(leaked_in_top5)}\n'
      f'  leakage_rate@5  = {leakage_at_5:.4f}   (lower = better; target 0.0)')



=== Ranking ===
  MRR             = 0.2461
  NDCG@5  (mean)  = 0.2508  (n=55)
  P@5     (mean)  = 0.1345
  R@10    (mean)  = 0.2817

=== Compliance ===
  BLOCK rate      = 0.2455  (2025 / 8250)
  REVIEW rate     = 0.0201
  leakage GT (total / testable) = 4 / 2
  violations leaked into top-5  = 2
  leakage_rate@5  = 1.0000   (lower = better; target 0.0)


## 6. Stratification — by `compliance_sensitivity` and `client_country`

AFCP-EM's distinguishing claim is *architectural* compliance, so the
stratified report should expose where the gate trades recall for safety.


In [7]:
rows = []
for row in ranked_rows:
    pid, ranked = row[0], row[1:]
    if pid not in ground_truth_rankings or not ground_truth_rankings[pid]:
        continue
    rel = set(ground_truth_rankings[pid])
    p = next((q for q in problems if q['problem_id'] == pid), {})
    rel_score_map = rel_scores_by_problem.get(pid, {eid: 1.0 for eid in rel})
    rows.append({
        'problem_id': pid,
        'client_country': (p.get('client_country') or '?'),
        'compliance_sensitivity': (p.get('compliance_sensitivity') or '?'),
        'technology_classification': (p.get('technology_classification') or '?'),
        'mrr': next((1.0 / pos for pos, e in enumerate(ranked, 1) if e in rel), 0.0),
        'ndcg_at_5': calculate_ndcg_at_k(ranked, rel_score_map, k=5),
        'precision_at_5': calculate_precision_at_k(ranked, rel, k=5),
        'recall_at_10': calculate_recall_at_k(ranked, rel, k=10),
        'pool_size_after_gate': sum(1 for r in row[1:] if r in expert_ids),
    })

df = pd.DataFrame(rows)
print(f'Evaluable rows: {len(df)} / {len(ranked_rows)} problems\n')
if df.empty:
    print('No problems with a non-empty relevant set — adjust the consensus threshold and re-run.')
else:
    cols = ['mrr', 'ndcg_at_5', 'precision_at_5', 'recall_at_10', 'pool_size_after_gate']
    print('=== by compliance_sensitivity ===')
    print(df.groupby('compliance_sensitivity')[cols].agg(['mean', 'count']).round(3))
    print('\n=== by client_country ===')
    print(df.groupby('client_country')[cols].agg(['mean', 'count']).round(3))
    print('\n=== by technology_classification ===')
    print(df.groupby('technology_classification')[cols].agg(['mean', 'count']).round(3))


Evaluable rows: 55 / 75 problems

=== by compliance_sensitivity ===
                          mrr       ndcg_at_5       precision_at_5        \
                         mean count      mean count           mean count   
compliance_sensitivity                                                     
confidential            0.332    28     0.252    28          0.150    28   
public                  0.262    20     0.226    20          0.100    20   
restricted              0.562     7     0.314     7          0.171     7   

                       recall_at_10       pool_size_after_gate        
                               mean count                 mean count  
compliance_sensitivity                                                
confidential                  0.337    28                 83.0    28  
public                        0.218    20                 83.0    20  
restricted                    0.242     7                 83.0     7  

=== by client_country ===
                  mrr 

## Notes

- **This is the floor.** TF-IDF + rule-based gate, no LLM, no embeddings. Beating this baseline is what the dissertation work has to do — the gap above this floor is the research contribution, not the absolute number.
- **Architectural compliance, by construction.** The gate masks blocked pairs out of `sim_row` *before* `argsort`, so a `leakage_rate@5 > 0` can only mean a rule miss (the gate didn't flag a true violation), never a post-hoc filter failure.
- **What L1 / L2 reveal about this floor.** `EXP_027` (kr_nationality=CN, kr_compliance_nct_flag=False) is marked as `S2 Clear Block` for `PROB_008` and `PROB_019` in the leakage GT, but the floor gate's R001 only checks the *expert's* NCT flag — it misses cases where the **problem's tech sensitivity + the expert's nationality** jointly trigger a block. Closing this is dissertation work; the floor honestly reports the gap.
- **Known data caveats.**
  - `data/problems.parquet` (SIRP, IDs `prob:000…`) and the curated ratings (IDs `PROB_001…`) are different problem sets — this notebook uses the SME problems for which ratings exist.
  - The ratings are on a **0–3 Likert** scale (not 1–5 as the original notebook spec claimed). This notebook uses ≥ 2 of 3 raters at score ≥ 2 as the consensus relevance threshold (265 positive pairs over 55 / 75 evaluable problems). A stricter threshold (≥ 2 / 3 raters at score ≥ 3) gives 39 pairs / 26 problems — useful as a high-confidence ablation.
  - The 3-rater GT has Fleiss κ = 0.258 / ICC(2,1) = 0.552 (moderate). Treat single-decimal metric differences as noise; report with the rater spread when used in papers.
  - Leakage GT is only 4 incidents — `leakage_rate@5` has low resolution. Adversarial-problem expansion is on the roadmap.
- **Out of scope here (intentional).** LLM-based keyword extraction, sentence-transformer / Bedrock embeddings, KGE scoring, LTR re-ranking, Tier-2 CoT reasoning, self-correction loop convergence — all out of scope for this floor baseline, which only needs to demonstrate the gate-then-score architecture.
- **Next steps when moving above the floor.** Replace TF-IDF with sentence-transformer embeddings (notebook 06?), keep this gate, re-run the same stratified report, and look for: (a) leakage_rate@5 staying at 0.0 even as recall rises, (b) widening gap between `client_country=CN` and `KR` pools (more strict gating → lower recall on restricted destinations — *expected and correct*).
